In [1]:
# %pip install pymupdf
# %pip install sentence-transformers
# %pip install faiss-cpu 
# %pip install lagchain
# %pip install Langchain-community 
# %pip install rouge-score
# %pip install nltk
# %pip install tqdm
# %pip install streamlit 

In [2]:
# %pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

In [3]:
import fitz          # reads PDFs
import json          # saves data in JSONL format
import os            # works with folders and files
import re            # cleans up messy text
from tqdm import tqdm  # shows a progress bar

In [4]:
RAW_DATA_FILE = "11-12th NCERT"
OUTPUT_FILE = "DATA/ncert_corpus.jsonl"

PREFIX_MAP = {
    "keph" : {"subject" : "Physics" , "class" : "11Th"},
    "kech" : {"subject" : "Chemistry" , "class" : "11Th"},
    "kebo" : {"subject" : "Biology" , "class" : "11Th"},
    "leph" : {"subject" : "Physics" , "class" : "12Th"},
    "lech" : {"subject" : "Chemistry" , "class" : "12Th"},
    "lebo" : {"subject" : "Biology" , "class" : "12Th"}
}

First one — reads the filename and figures out subject, class and chapter

In [5]:
def parse_pdf(filename):
    name = filename.lower().replace(".pdf" , "")
    for prefix,info in PREFIX_MAP.items():
        if name.startswith(prefix):
            chapter_part = name[len(prefix):] # get the part after the prefix
            chapter = int(chapter_part[-2:]) # get the last two characters as chapter number
            return info["subject"] ,info["class"] , chapter 
    return None, None, None  # return None if no prefix matches

Second one — opens a PDF and pulls out the text

In [6]:
def clean_text(text):
    text = re.sub(r'\s+'," ",text)  # replace multiple whitespace with single space
    text = re.sub(r"[^\x00-\x7F]+"," ",text)  # remove non-ASCII characters
    text = text.strip()
    return text

def extract_text_from_pdf(pdf_path): # extracts text from a PDF file and returns a list of cleaned text for each page
    doc = fitz.open(pdf_path)
    pages_text = []
    for page in doc:
        text = page.get_text("text")
        text = clean_text(text)
        if len(text) > 100:
            pages_text.append(text)
    doc.close()
    return pages_text

In [7]:
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if len(chunk) > 50:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def main():
    os.makedirs("DATA", exist_ok=True)

    pdf_files = [i for i in os.listdir(RAW_DATA_FILE) if i.lower().endswith(".pdf")]
    print(f"Found {len(pdf_files)} PDFs\n")

    skipped = []
    total_chunks = 0

    with open(OUTPUT_FILE, "w", encoding="utf-8") as out_file:
        for filename in tqdm(sorted(pdf_files), desc="Processing PDFs"):
            subject, class_num, chapter = parse_pdf(filename)

            if subject is None:
                skipped.append(filename)
                continue

            filepath = os.path.join(RAW_DATA_FILE, filename)
            pages = extract_text_from_pdf(filepath)
            full_text = " ".join(pages)
            chunks = chunk_text(full_text)

            for chunk_num, chunk in enumerate(chunks):
                record = {
                    "subject":  subject,
                    "class":    class_num,
                    "chapter":  chapter,
                    "source":   filename.replace(".pdf", ""),
                    "chunk":    chunk_num,
                    "text":     chunk
                }
                out_file.write(json.dumps(record) + "\n")
                total_chunks += 1

    print(f"\n '✅ Done!")
    print(f"   Total chunks extracted : {total_chunks}")
    print(f"   Saved to               : {OUTPUT_FILE}")

    if skipped:
        print(f"\n ☢️ Skipped {len(skipped)} unrecognized files : {skipped}")

main()

Found 66 PDFs



Processing PDFs: 100%|██████████| 66/66 [00:07<00:00,  9.20it/s]


 '✅ Done!
   Total chunks extracted : 4048
   Saved to               : DATA/ncert_corpus.jsonl


# phase 2

step-1 imports

In [8]:
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

C:\Users\ayush\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


step-2 configuration

In [9]:
CORPUS_FILE = "DATA/ncert_corpus.jsonl"
INDEX_FILE = "DATA/ncert_index.faiss"
METADATA_FILE = "DATA/ncert_metadata.json"
MODEL_NAME = "all-MiniLM-L6-v2"  # You can change this to any other model from sentence-transformers

step-3 loading corpus

In [10]:
texts = []
metadata = []

with open(CORPUS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        texts.append(record["text"])
        metadata.append({
            "subject": record["subject"],
            "class":   record["class"],
            "chapter": record["chapter"],
            "source":  record["source"],
            "chunk":   record["chunk"]
        })

print(f"Loaded {len(texts)} entries")

Loaded 4048 entries


step-4 Loading and embedding model

In [11]:
print(f"Loading model{MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print("Model loaded successfully")

Loading modelall-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11205.70it/s]


Model loaded successfully


step-5 Generating embeddings

In [12]:
print("Generating embeddings...")
embeddings = model.encode(texts, show_progress_bar = True , batch_size = 64)
print(f"embeddings shape : {embeddings.shape}")

Generating embeddings...


Batches: 100%|██████████| 64/64 [00:59<00:00,  1.08it/s]

embeddings shape : (4048, 384)


part-6 Building and saving FAISS index

In [13]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

faiss.write_index(index, INDEX_FILE)

with open(METADATA_FILE , "w" , encoding="utf-8") as f:
    json.dump(metadata,f) 

print(f"Index built with {index.ntotal} vectors")
print(f"Saved index to {INDEX_FILE}")
print(f"Saved matdata to {METADATA_FILE}")

Index built with 4048 vectors
Saved index to DATA/ncert_index.faiss
Saved matdata to DATA/ncert_metadata.json


# phase 3
part-1 Imports

In [14]:
from llama_cpp import Llama
import faiss
import json
import numpy as np
from sentence_transformers import SentenceTransformer

part-2 Configuration

In [25]:
INDEX_FILE    = "data/ncert_index.faiss"
METADATA_FILE = "data/ncert_metadata.json"
CORPUS_FILE   = "data/ncert_corpus.jsonl"
MODEL_PATH    = "llama-2-7b-chat.Q4_K_M.gguf"
MODEL_NAME    = "all-MiniLM-L6-v2"

TOP_K = 5  # number of relevant pages to retrieve per question

part-3 loading everything

In [26]:
index = faiss.read_index(INDEX_FILE)

with open(METADATA_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)

texts = []
with open(CORPUS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        texts.append(record["text"])

embedder = SentenceTransformer(MODEL_NAME)

print("FAISS index loaded:", index.ntotal, "vectors")
print("Metadata loaded:", len(metadata), "entries")
print("Corpus loaded:", len(texts), "pages")
print("Embedding model ready")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20559.34it/s]


FAISS index loaded: 4048 vectors
Metadata loaded: 4048 entries
Corpus loaded: 4048 pages
Embedding model ready


step-4 loadind llama 2

In [27]:
print("Loading Llama-2 model...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2024,       
    n_gpu_layers=32,   
    verbose=False      
)
print("Llama-2 ready!")

Loading Llama-2 model...


llama_context: n_ctx_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


Llama-2 ready!


step-5 the retriver fxn

In [28]:
def retrieve(question, top_k=TOP_K):

    question_vector = embedder.encode([question])
    
    distances, indices = index.search(np.array(question_vector), top_k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "text":     texts[idx],
            "subject":  metadata[idx]["subject"],
            "class":    metadata[idx]["class"],
            "chapter":  metadata[idx]["chapter"],
            "source":   metadata[idx]["source"],
            "chunk":    metadata[idx]["chunk"],
            "score":    float(distances[0][i])
        })
    
    return results

part-6 the answer generation fxn

In [39]:
def generate_answer(question, retrieved_pages):

    context = ""
    for i, page in enumerate(retrieved_pages):
        context += f"[Source {i+1}: {page['subject']} Class {page['class']} Chapter {page['chapter']}]\n"
        context += page['text'][:600] + "\n\n"

    prompt = f"""[INST] <<SYS>>
You are an NCERT tutor for Class 11 and 12 students. Answer the question using ONLY the context below. Be direct and concise. Do not say you cannot find the answer if the information is present in the context.
<</SYS>>

Context:
{context}

Question: {question} [/INST]"""

    response = llm(
        prompt,
        max_tokens=512,
        temperature=0.2,
        stop=["</s>", "[INST]"]
    )

    return response["choices"][0]["text"].strip()

part-7 putting it all together main RAG fxn

In [30]:
def ask(question):
    print(f"\n Question: {question}")
    print("-" * 50)
    
    # retrieve relevant pages
    retrieved_pages = retrieve(question)
    
    # generate answer
    answer = generate_answer(question, retrieved_pages)
    
    print(f" Answer:\n{answer}")
    print("\n Sources:")
    for i, page in enumerate(retrieved_pages):
        print(f"  {i+1}. {page['subject']} | Class {page['class']} | Chapter {page['chapter']} | Score: {page['score']:.2f}")
    
    return answer

In [40]:
test_questions = [
    {
        "question": "What is photosynthesis?",
        "reference": "Photosynthesis is a process used by green plants and other organisms to convert sunlight, water, and carbon dioxide into food and oxygen."
    },
    {
        "question": "State Newton's second law of motion.",
        "reference": "Newton's second law states that the force acting on an object is equal to the mass of the object multiplied by its acceleration. F equals ma."
    },
    {
        "question": "What is mitosis?",
        "reference": "Mitosis is a type of cell division in which a single cell divides to produce two identical daughter cells with the same number of chromosomes as the parent cell."
    },
    {
        "question": "What is Ohm's law?",
        "reference": "Ohm's law states that the current through a conductor is directly proportional to the voltage across it and inversely proportional to its resistance. V equals IR."
    },
    {
        "question": "What are covalent bonds?",
        "reference": "Covalent bonds are chemical bonds formed by the sharing of electron pairs between atoms. They typically occur between non-metal atoms."
    },
    {
        "question": "What is the structure of DNA?",
        "reference": "DNA is a double helix structure made of two strands of nucleotides. Each nucleotide contains a sugar, a phosphate group, and a nitrogenous base. The bases pair specifically: adenine with thymine and guanine with cytosine."
    },
    {
        "question": "What is the law of conservation of energy?",
        "reference": "The law of conservation of energy states that energy cannot be created or destroyed, only converted from one form to another. The total energy of an isolated system remains constant."
    },
    {
        "question": "What are enzymes?",
        "reference": "Enzymes are biological catalysts that speed up chemical reactions in living organisms without being consumed in the process. They are mostly proteins and are highly specific to their substrates."
    },
    {
        "question": "What is electromagnetic induction?",
        "reference": "Electromagnetic induction is the process of generating an electric current by changing the magnetic field around a conductor. It was discovered by Michael Faraday."
    },
    {
        "question": "What is chemical equilibrium?",
        "reference": "Chemical equilibrium is the state in which the rate of the forward reaction equals the rate of the reverse reaction, so the concentrations of reactants and products remain constant over time."
    },
]

print(f"Total test questions: {len(test_questions)}")

Total test questions: 10


In [41]:
import csv
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ayush\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [42]:
def evaluate(generated, reference):

    reference_tokens = [reference.lower().split()]
    generated_tokens = generated.lower().split()
    smoothing = SmoothingFunction().method1
    bleu = sentence_bleu(reference_tokens, generated_tokens, smoothing_function=smoothing)
    
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge = scorer.score(reference, generated)
    rouge_l = rouge['rougeL'].fmeasure
    
    return round(bleu, 4), round(rouge_l, 4)

In [43]:
results = []

print("Running evaluation...\n")

for i, item in enumerate(test_questions):
    question = item["question"]
    reference = item["reference"]
    
    print(f"Q{i+1}: {question}")
    
    retrieved_pages = retrieve(question)
    generated = generate_answer(question, retrieved_pages)
    
    # score it
    bleu, rouge_l = evaluate(generated, reference)
    
    print(f"Answer: {generated[:100]}...")
    print(f"BLEU: {bleu} | ROUGE-L: {rouge_l}\n")
    
    results.append({
        "question":   question,
        "reference":  reference,
        "generated":  generated,
        "bleu":       bleu,
        "rouge_l":    rouge_l
    })

print("✅ Evaluation complete!")

Running evaluation...

Q1: What is photosynthesis?
Answer: Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy f...
BLEU: 0.0111 | ROUGE-L: 0.25

Q2: State Newton's second law of motion.
Answer: Newton's second law of motion is given by:
F = ma

Where F is the net external force on the body, m ...
BLEU: 0.1062 | ROUGE-L: 0.4127

Q3: What is mitosis?
Answer: Mitosis is the equational division of a cell, resulting in the production of two daughter cells with...
BLEU: 0.1111 | ROUGE-L: 0.297

Q4: What is Ohm's law?
Answer: Ohm's law is a fundamental principle in physics that relates the voltage (V), current (I), and resis...
BLEU: 0.1622 | ROUGE-L: 0.4364

Q5: What are covalent bonds?
Answer: Covalent bonds are bonds that are formed by the sharing of electrons between two atoms. In a covalen...
BLEU: 0.0989 | ROUGE-L: 0.3784

Q6: What is the structure of DNA?
Answer: Based on the information provided in Sources 1-5, the structure of DNA is as

In [46]:
import csv
import os

os.makedirs("../outputs", exist_ok=True)

with open("../outputs/evaluation.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "reference", "generated", "bleu", "rouge_l"])
    writer.writeheader()
    writer.writerows(results)

print("✅ Saved to outputs/evaluation.csv")

✅ Saved to outputs/evaluation.csv
